In [ ]:
%pip install scikit-learn
%pip install opencv-python
%pip install pandas
%pip install torch torchvision torchaudio
%pip install matplotlib

In [ ]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
import torchvision.models as models

import os
# import kagglehub

from PIL import Image
import pandas as pd
from confidence_functions import *

from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score
import matplotlib.pyplot as plt


In [ ]:
import socket
print(socket.gethostname())

# use all available cpu nodes
torch.set_num_threads(os.cpu_count())

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Download data if have not
# path = kagglehub.dataset_download("thedrcat/hpa-cell-tiles-sample-balanced-dataset")

# print("Path to dataset files:", path)

In [ ]:
# csv_path = os.path.join(path, "cell_df.csv")
# cell_dir = os.path.join(path, "cells")
csv_path = '/home/ykc0662/.cache/kagglehub/datasets/thedrcat/hpa-cell-tiles-sample-balanced-dataset/versions/1/cell_df.csv'
cell_dir = '/home/ykc0662/.cache/kagglehub/datasets/thedrcat/hpa-cell-tiles-sample-balanced-dataset/versions/1/cells'

df = pd.read_csv(csv_path, usecols=['image_id', 'cell_id', 'image_labels'])


In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # distorts aspect ratio, but simple & fast
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# train model - stability calculated with only the last 100 trainings
def train_model(n_epochs, batch_size, amount_data, data=hypersphere_data,
                cat100=True, output_dim=19):
    train_loader = data(input_dim, output_dim, amount_data, 
                                    batch_size = batch_size, seed = seed)    
    
    # use pretrained resnet18 model
    model = models.resnet18(weights='IMAGENET1K_V1')

    # freeze everything except the last residual block and the final FC layer
    for name, param in model.named_parameters():
        if not (name.startswith('layer4') or name.startswith('fc')):
            param.requires_grad = False

    # change output layer to have 19 logits
    model.fc = nn.Linear(512, 19)
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-1)

    # cat_last = None 
    # perc_change = [] 
    cat_all = torch.zeros(len(X), output_dim).to(device)
    loss_history = []


    total_iterations = n_epochs * (amount_data//batch_size) 
    target_iteration = total_iterations - 100 # start calculating cat_all for the last 100 iterations

    itr = 0

    for _ in range(n_epochs):
        for data, target in train_loader:
            itr += 1
            data, target = data.to(device), target.to(device)

            outputs = model(data)
            loss = criterion(outputs, target)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_history.append(loss.item())

            with torch.no_grad():
                cat_next = model(X).argmax(dim=1)
                # if cat_last != None:
                #     perc_change.append( (cat_last != cat_next).sum().item() / len(X) )
                # cat_last = cat_next

                if cat100==False:
                    for i in range(output_dim):
                        cat_all[cat_next == i, i] += 1
                else:
                    if itr>=target_iteration:
                        for i in range(output_dim):
                            cat_all[cat_next == i, i] += 1
                    
    # Calculate your stability based on the rolling 100-step history (or not)
    stab = shanon_stability(cat_all, output_dim)

    # calculate means and variances for mahalanobis
    stats = compute_class_means_and_covariance(model, train_loader, output_dim , device)

    return model, stab, loss_history, stats

In [ ]:
class CellDataset(Dataset):
    def __init__(self, image_paths, labels, transform):
        self.image_paths = image_paths
        self.labels = labels  # multi-hot vectors
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.int8)
        return img, label

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)